<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/12_evaluators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12 · Evaluators that live in LangSmith

An evaluator is a function that looks at what the agent did and returns a score. The interesting
question is not how to write one — it is **where it lives**.

Write it in this notebook and it dies with the kernel. Create it in LangSmith and it becomes a
workspace object: named, versioned, attachable to a dataset or to live traffic, and usable by
people who have never seen your code.

**New in this lesson:** the evaluators API, `perform_eval`, stored LLM-as-judge prompts,
attaching evaluators with rules, and retro-scoring an experiment

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-12-evaluators"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Connect to the shared deployment (run me) { display-mode: "form" }
# --- snippet:remote_agent v1 ---
import hashlib

import httpx
from langgraph.pregel.remote import RemoteGraph

HOST_API = "https://api.host.langchain.com"
LS_API = "https://api.smith.langchain.com/api/v1"
HEADERS = {"x-api-key": key}


def find_deployment(name: str = "support-agent") -> dict:
    """The shared deployment's record, looked up by name."""
    response = httpx.get(
        f"{HOST_API}/v2/deployments",
        params={"name_contains": name},
        headers={"X-Api-Key": os.environ["LANGSMITH_API_KEY"]},
        timeout=30,
    )
    response.raise_for_status()
    for record in response.json()["resources"]:
        if record["name"] == name:
            return record
    raise RuntimeError(f"No deployment named {name!r} in this workspace.")


DEPLOYMENT = find_deployment()

# Every attendee's calls land in this one project. That is the point: shared traffic.
TRAFFIC_PROJECT_ID = DEPLOYMENT["tracer_session_id"]

# "support" is the graph key from langgraph.json in lesson 10.
support = RemoteGraph("support", url=DEPLOYMENT["url"], api_key=key)


def last_text(result: dict) -> str:
    """The final reply. A deployment returns JSON, so messages are dicts."""
    content = result["messages"][-1].get("content") or ""
    if isinstance(content, list):
        # The model returns reasoning blocks alongside the answer; keep the answer.
        return " ".join(part["text"] for part in content
                        if isinstance(part, dict) and part.get("type") == "text")
    return str(content)


def tool_names(result: dict) -> list[str]:
    """Every tool the run called, in order."""
    return [call["name"]
            for message in result["messages"]
            for call in (message.get("tool_calls") or [])]


# Everyone shares the agent; nobody shares your datasets. Derived from your key so
# it is unique to you and the same every time you run this.
ME = hashlib.sha256(key.encode()).hexdigest()[:8]
# --- /snippet ---

print(f"{DEPLOYMENT['name']}: {DEPLOYMENT['status']} | you are {ME}")

In [ ]:
from langsmith import Client

client = Client()

In [ ]:
# The dataset from lesson 11. If you skipped it, this creates a small one.
name = f"refund-decisions-{ME}"
if client.has_dataset(dataset_name=name):
    dataset = client.read_dataset(dataset_name=name)
else:
    dataset = client.create_dataset(name, description="Refund decisions with a clear right answer.")
    client.create_examples(dataset_id=dataset.id, examples=[
        {"inputs": {"messages": [{"role": "user", "content":
            "Ticket T-6: the laptop stand on order 1047 wobbles. What can we offer?"}],
                    "question": "Ticket T-6: the laptop stand on order 1047 wobbles."},
         "outputs": {"decision": "repair", "because": "Faulty after 30 days is repair only."}},
        {"inputs": {"messages": [{"role": "user", "content":
            "Order 1042 arrived with a cracked leg. The customer wants a full refund."}]},
         "outputs": {"decision": "refund", "because": "Damaged on arrival: full refund, no time limit."}},
        {"inputs": {"messages": [{"role": "user", "content":
            "I ordered the wrong colour chair, order 1045. Can I swap it?"}]},
         "outputs": {"decision": "exchange", "because": "Customer error: exchange within 14 days."}},
    ])

print(dataset.name, "|", len(list(client.list_examples(dataset_id=dataset.id))), "examples")

---

## 1. Two homes for an evaluator

Both are legitimate. They are good at different things.

| | **In your code** | **In LangSmith** |
|---|---|---|
| Lives in | your repo, your test suite | the workspace |
| Runs | when you run it | whenever a rule fires |
| Reviewed by | code review | anyone with a browser |
| Can score | experiments you launch | experiments **and live traffic** |
| Needs | the notebook to be running | nothing |
| Good for | logic tied to your app's internals | shared standards, online evals, non-engineers |

The one that matters most in practice is the second-to-last row. An evaluator in your code cannot
score production traffic at 3am, because nothing is running it. An evaluator in LangSmith can.

This lesson builds evaluators the second way. The first way is a few lines you already know how to
write, and lesson 15 uses it where it belongs — in CI.

---

## 2. A code evaluator, stored

The contract is one function called `perform_eval`, taking a run and an example, returning a dict
of feedback. Keys become feedback names; values become scores.

In [ ]:
POLICY_CHECK = r"""
def perform_eval(run, example):
    # Did the agent land on the decision the refund policy requires?
    import re

    expected = (example["outputs"] or {}).get("decision", "")
    messages = (run["outputs"] or {}).get("messages") or []
    answer = (messages[-1].get("content") if messages else "") or ""
    if isinstance(answer, list):
        answer = " ".join(part["text"] for part in answer
                          if isinstance(part, dict) and part.get("type") == "text")
    answer = answer.lower()

    words = {
        "refund": ("refund", "money back"),
        "repair": ("repair", "fix"),
        "exchange": ("exchange", "swap", "replace"),
        "escalate": ("escalate", "escalation"),
    }
    said = {name for name, terms in words.items() if any(t in answer for t in terms)}

    # Refunds over $200 need human approval, so a large refund has to say so.
    amounts = [float(a) for a in re.findall(r"\$([0-9]+(?:\.[0-9]{2})?)", answer)]
    big_refund = "refund" in said and any(a > 200 for a in amounts)
    approved = not big_refund or "approval" in answer or "approve" in answer

    return {
        # Membership, not equality: a reply often names another outcome to rule it out.
        "decision_correct": 1 if expected in said else 0,
        "approval_respected": 1 if approved else 0,
    }
"""

created = httpx.post(
    f"{LS_API}/platform/evaluators",
    headers=HEADERS,
    json={
        "name": f"refund-decision-check-{ME}",
        "type": "code",
        "code_evaluator": {"code": POLICY_CHECK, "language": "python"},
    },
    timeout=60,
).json()["evaluator"]

code_evaluator_id = created["id"]
print("id:", code_evaluator_id)
print("feedback keys:", created["feedback_keys"])

Three things happened there worth naming.

**A plain HTTP call.** Evaluators are a workspace resource, so this is a `POST` like any other. The
Python SDK also wraps it as `await client.evaluators.create(...)`, which is tidier in application
code; the notebooks use HTTP because it shows you exactly what the resource is and works the same
from any language.

**Feedback keys were derived from your code.** You did not declare `decision_correct` and `hedged`
anywhere — LangSmith read the returned dict. Which means a typo in a key silently creates a new
metric rather than failing, so check that list.

**One correctness check and one safety check.** `decision_correct` asks only whether the right
outcome is *present* — it has to, because a good reply often names another outcome in order to rule
it out ("a repair only, not a refund"). Insist on an exact match and you punish the most careful
answers, which is worse than having no evaluator: you would spend a day debugging the agent before
thinking to suspect the test.

`approval_respected` is the shape worth copying. It is **conditional** — it only bites when the
reply actually promises a refund over $200 — and it encodes a rule from the business rather than a
habit of the current agent. Those age well. Contrast it with a plausible-sounding metric like "the
reply mentions the policy", which passes every time on this agent because its skill tells it to: an
assertion that cannot fail is not testing anything.

### What runs inside the sandbox

Your code executes on LangSmith's infrastructure, not yours. That comes with limits:

- **Imports:** the standard library, plus `numpy`, `pandas`, `scipy`, `scikit-learn`, `jsonschema`.
- **No network.** No calling your API to look something up.
- **`run` and `example` are dicts**, not the SDK's `Run` and `Example` objects — so it is
  `run["outputs"]`, never `run.outputs`.

That last one is the mistake everyone makes once. Test the logic locally on a real run first: an
evaluator that throws produces no feedback, and a metric that is quietly absent is worse than one
that is wrong.

In [ ]:
# Test it locally before trusting it. Same code, called by hand on a real run.
namespace = {}
exec(POLICY_CHECK, namespace)

fake_run = {"outputs": {"messages": [{"content": "We can offer a repair for the wobbly stand."}]}}
fake_example = {"outputs": {"decision": "repair"}}
print(namespace["perform_eval"](fake_run, fake_example))

hedging_run = {"outputs": {"messages": [{"content": "We could repair it, or refund, or exchange."}]}}
print(namespace["perform_eval"](hedging_run, fake_example))

---

## 3. Attaching it

An evaluator on its own scores nothing. A **rule** says what it watches: a dataset (score every
experiment run against it) or a tracing project (score live traffic, which is lesson 14).

In [ ]:
rule = httpx.post(
    f"{LS_API}/runs/rules",
    headers=HEADERS,
    json={
        "display_name": f"score-refund-decisions-{ME}",
        "dataset_id": str(dataset.id),          # every experiment on this dataset
        "evaluator_id": str(code_evaluator_id),
        "sampling_rate": 1.0,
        "is_enabled": True,
    },
    timeout=30,
).json()

rule_id = rule["id"]
print("rule:", rule["display_name"], rule_id)

---

## 4. Run the experiment

Now the part that looks like a trick. Run an experiment against the deployed agent and pass
**no evaluators at all**.

In [ ]:
def run_agent(inputs: dict) -> dict:
    """The experiment target. Returns the answer as a string as well as the raw
    messages, because an evaluator can only map a variable onto a scalar."""
    result = support.invoke(inputs)
    return {"answer": last_text(result), "messages": result["messages"]}


results = client.evaluate(
    run_agent,
    data=dataset.name,
    experiment_prefix=f"stored-evaluator-{ME}",
    max_concurrency=2,
    evaluators=[],                 # deliberately empty
)

print(results.experiment_name)

Open that experiment in LangSmith. The scores appear anyway, a few seconds behind the runs, because
the rule is attached to the dataset and fired on its own.

That is the shape of the whole idea:

```
dataset ──> experiment ──> rule fires ──> feedback
                             ▲
                         evaluator
```

Your notebook's job was to produce runs. Grading them is somebody else's — which is why grading
keeps happening after you close the tab.

Two consequences worth planning for. Scores arrive **asynchronously**, so a script that reads the
average immediately after `evaluate()` returns may read it too early — which is why the cell below
polls. Note that the runs query returns **only ids** unless you name the fields you want in
`select`. And a rule is attached to the
**dataset**, not to your experiment, so a colleague running their own experiment on the same
dataset gets the same scoring for free — which is the point, and also means changing the evaluator
changes everyone's numbers.

In [ ]:
# Wait for the feedback to land, then read the scores back.
import time

experiment = client.read_project(project_name=results.experiment_name)

for attempt in range(24):
    runs = httpx.post(
        f"{LS_API}/runs/query",
        headers=HEADERS,
        json={"session": [str(experiment.id)], "is_root": True,
              "select": ["id", "feedback_stats"]},
        timeout=60,
    ).json()["runs"]
    scored = [run for run in runs if run.get("feedback_stats")]
    if scored:
        break
    time.sleep(5)

print(f"{len(scored)}/{len(runs)} runs scored\n")
if not scored:
    print("Scores have not landed yet. The rule runs behind the experiment; "
          "re-run this cell, or just look at the experiment in LangSmith.")
for run in scored:
    for feedback_key, stats in run["feedback_stats"].items():
        print(f"  {feedback_key:20} avg={stats.get('avg')}  n={stats.get('n')}")

---

## 5. An LLM-as-judge, stored

Some things cannot be checked with `in`. "Did the agent cite the policy line it relied on?" needs a
reader.

A stored LLM evaluator is **a prompt in the LangSmith prompt hub plus a mapping**. That is a design
decision worth noticing: your rubric becomes a versioned artifact with a diff history, rather than
a string buried in a Python file.

The prompt has to be a `StructuredPrompt` — a template with an output schema — because the judge
must return fields, not prose.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts.structured import StructuredPrompt
from langsmith.utils import LangSmithConflictError

VERDICT_SCHEMA = {
    "title": "policy_verdict",
    "description": "Whether the agent justified its decision with the refund policy.",
    "type": "object",
    "properties": {
        "cites_policy": {
            "type": "boolean",
            "description": "True if the answer states the policy rule it relied on, not just the outcome.",
        },
        "comment": {"type": "string", "description": "One sentence explaining the verdict."},
    },
    "required": ["cites_policy", "comment"],
}

judge_prompt = StructuredPrompt.from_messages_and_schema(
    [
        ("system",
         "You grade a customer support agent's reply.\n"
         "The agent must not only reach the right outcome, it must say which policy rule got it "
         "there, so a human can check the reasoning.\n"
         "Reward a reply that names the rule. Do not reward a confident reply with no rule cited."),
        ("human",
         "Customer question:\n{input}\n\nAgent reply:\n{output}\n\nCorrect decision:\n{reference}"),
    ],
    VERDICT_SCHEMA,
)

# The pushed object has to include a model. A bare prompt is a one-step chain, and
# LangSmith rejects it with "RunnableSequence must have at least 2 steps, got 0".
judge_chain = judge_prompt | init_chat_model(MODEL)

handle = f"refund-policy-judge-{ME}"
try:
    client.push_prompt(handle, object=judge_chain, description="Does the reply cite the policy rule?")
except LangSmithConflictError:
    pass          # unchanged since the last push, so re-running this cell is fine

pushed = client.get_prompt(handle)
assert pushed and pushed.last_commit_hash, "prompt was not found after pushing"

prompt_handle, commit = pushed.repo_handle, pushed.last_commit_hash
print(prompt_handle, "| commit:", commit[:12])

In [ ]:
judge = httpx.post(
    f"{LS_API}/platform/evaluators",
    headers=HEADERS,
    json={
        "name": f"cites-policy-{ME}",
        "type": "llm",
        "llm_evaluator": {
            "prompt_repo_handle": prompt_handle,
            "commit_hash_or_tag": commit,
            # Left side: the prompt's variables. Right side: where to find them on the run.
            # Singular roots, and each has to land on a scalar: a list comes
            # back as null, silently.
            "variable_mapping": {
                "input": "input.question",
                "output": "output.answer",
                "reference": "reference.decision",
            },
        },
    },
    timeout=60,
).json()["evaluator"]

print("id:", judge["id"], "| feedback keys:", judge["feedback_keys"])

`feedback_keys` came back as `cites_policy` and not `comment`. The schema's boolean became a
**score**; the string became the comment attached to it. That is how you design a judge schema: one
field to measure, one field to explain, and the explanation is what you read when the measurement
surprises you.

`variable_mapping` is the join between two things that know nothing about each other: a prompt with
`{input}`, `{output}`, `{reference}`, and a run that has inputs, outputs and a reference. Two rules
about it are easy to get wrong and fail **silently**, with the variable arriving as `null` and the
judge grading nothing:

- the roots are **singular** — `input.`, `output.`, `reference.`, not `inputs.`/`outputs.`
- each has to land on a **scalar**. `output.messages` is a list, so it maps to `null`; that is why
  the target above returns `answer` as a string.

Pin the `commit_hash_or_tag` and the rubric is frozen; point it at a tag like `production` and you
can improve the judge without touching the evaluator.

### One rubric, two places it can run

The rubric is an ordinary prompt, so you can run it yourself. Same wording, same schema, same
verdict a server-side judge would reach — which makes it the fastest way to iterate on a rubric
before anything is stored.

In [ ]:
reply = last_text(support.invoke({"messages": [{"role": "user", "content":
    "Ticket T-6: the laptop stand on order 1047 wobbles. What can we offer?"}]}))

# `with_structured_output` binds the schema as a tool call, which is the reliable way
# to get fields back. The prompt carries the same schema for LangSmith's benefit.
local_judge = init_chat_model(MODEL).with_structured_output(VERDICT_SCHEMA)
verdict = local_judge.invoke(judge_prompt.format_messages(
    input="Ticket T-6: the laptop stand on order 1047 wobbles.",
    output=reply,
    reference="repair",
))
print(verdict)

> **Before LangSmith can run it for you:** an LLM evaluator needs a **model configuration** in the
> workspace — that is what the `playground_settings_id` field on the evaluator refers to. Without
> one, the evaluator is created happily and then fails at grading time with
> `OpenAIPermissionDeniedError: 403`, because the server has no credentials it is allowed to use for
> your model. Create one under **Settings → Model configurations**, or keep judging locally as
> above. Code evaluators have no such prerequisite, which is another reason to reach for them first.

---

## 6. Scoring an experiment you already ran

The awkward truth about evaluators is that you usually think of the right one **after** you have
seen the results. You do not have to re-run the agent to apply it.

In [ ]:
LENGTH_CHECK = r"""
def perform_eval(run, example):
    # Short enough to send to a customer without editing?
    messages = (run["outputs"] or {}).get("messages") or []
    answer = (run["outputs"] or {}).get("answer") or ""
    if not answer and messages:
        content = messages[-1].get("content") or ""
        answer = content if isinstance(content, str) else " ".join(
            p["text"] for p in content if isinstance(p, dict) and p.get("type") == "text")
    words = len(str(answer).split())
    return {"sendable_length": 1 if words <= 120 else 0, "word_count": words}
"""

length = httpx.post(
    f"{LS_API}/platform/evaluators",
    headers=HEADERS,
    json={"name": f"sendable-length-{ME}", "type": "code",
          "code_evaluator": {"code": LENGTH_CHECK, "language": "python"}},
    timeout=60,
).json()["evaluator"]

# Attach it to the dataset...
length_rule = httpx.post(
    f"{LS_API}/runs/rules",
    headers=HEADERS,
    json={
        "display_name": f"score-length-{ME}",
        "dataset_id": str(dataset.id),
        "evaluator_id": length["id"],
        "sampling_rate": 1.0,
        "is_enabled": True,
    },
    timeout=30,
).json()

# ...then point it at the experiment that has already finished.
applied = httpx.post(
    f"{LS_API}/runs/experiments/{experiment.id}/evaluate",
    headers=HEADERS,
    json={"rule_id": length_rule["id"]},
    timeout=60,
)
print(applied.status_code, "|", length["feedback_keys"])

The same runs now carry two more metrics, added after the fact. No agent calls, no new experiment —
the outputs were already stored, and an evaluator only ever reads stored outputs.

`word_count` is worth noticing: it is not a pass or a fail, it is a number you can chart across
experiments. Useful metrics are not all assertions.

This is the strongest practical argument for keeping evaluators in the workspace. Six weeks from
now, when someone asks "how often did it cite the policy?", you can answer for every experiment you
have ever run instead of for the ones where you happened to have the right assertion in your test
file.

---

## 7. Which kind, and where

Reach for a **code evaluator** when the answer is decidable: a field exists, a number is in range,
JSON parses, a forbidden string is absent, latency is under budget. It is free, instant, and
deterministic. Most suites should be mostly these, and most teams have far too few of them.

Reach for an **LLM judge** when the property is genuinely a judgement: tone, faithfulness to a
source, whether reasoning follows. Expect to spend real effort on it — a judge is a model in
production and it needs its own evidence that it works, which is what lesson 14's alignment loop is
for.

Keep it **in your repo** when the check depends on your internals — a database fixture, a private
helper, the shape of your own config. Do not contort a sandboxed evaluator to reach something it
cannot see.

And a warning that costs people money: an LLM judge attached to a busy project at
`sampling_rate: 1.0` is an LLM call per trace, forever. That is a bill, not a rounding error.

---

## 📌 Key takeaways

- An evaluator stored in LangSmith is a workspace object: named, attachable, and alive when your notebook is not.
- Code evaluators are one function, `perform_eval(run, example)`, returning `{feedback_key: score}`.
- Feedback keys are **derived from your returned dict**, so a typo invents a metric instead of failing.
- Inside the sandbox, `run` and `example` are dicts — `run["outputs"]`, never `run.outputs`.
- The sandbox has the stdlib plus numpy/pandas/scipy/sklearn, and **no network**.
- An evaluator that raises produces silent nothing, not a zero. Test it locally first.
- A **rule** binds an evaluator to a dataset (experiments) or a project (live traffic).
- With a rule attached, `evaluate(..., evaluators=[])` still gets scored — grading is no longer your notebook's job.
- Scores land asynchronously, so do not read the average the instant `evaluate()` returns.
- A stored LLM judge is a **`StructuredPrompt` in the hub** plus a `variable_mapping` — the rubric gets version history.
- In a judge schema, a boolean becomes the score and a string becomes the comment you read when the score surprises you.
- You can score an experiment that already finished — evaluators read stored outputs, so no re-run is needed.
- Prefer deterministic code evaluators; spend LLM judges on properties that are genuinely judgement calls.

---

## ➡️ Next

**[13 · Testing the path, not just the answer](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/13_trajectory_evals.ipynb)**

Both evaluators so far read only the final answer. But an agent can reach the right answer for the
wrong reasons — guessing the policy instead of reading it. Next: scoring the **path**.